In [ ]:
# 1. Install the NEW Google GenAI library
!pip install -q -U google-genai tqdm

import json
import pandas as pd
from tqdm import tqdm
from google import genai
from google.colab import drive, userdata, files

# 2. Mount Drive
drive.mount('/content/drive')

# 3. Setup Gemini API 
# IMPORTANT: In the Colab 'Secrets' (key icon), make sure you have 
# a secret named GEMINI_API_KEY with your actual key as the value.
try:
    # Use the LABEL name here, not the actual 'AIza...' string
    api_key = userdata.get('GEMINI_API_KEY') 
    client = genai.Client(api_key=api_key)
    print("Gemini API configured successfully using the new SDK.")
except Exception as e:
    print(f"Error: Make sure you added 'GEMINI_API_KEY' to Colab Secrets. {e}")

In [ ]:
def query_gemini_agent(system_prompt, user_message):
    """Sends a request to Gemini using the new SDK and returns parsed JSON."""
    full_prompt = f"{system_prompt}\n\nInput Data:\n{user_message}"
    
    try:
        # Use 'gemini-1.5-flash' for speed/cost or 'gemini-1.5-pro' for high intelligence
        response = client.models.generate(
            model="gemini-1.5-flash",
            contents=full_prompt,
            config={
                'response_mime_type': 'application/json',
            }
        )
        # The new SDK returns the text directly in response.text
        return json.loads(response.text)
    except Exception as e:
        print(f"Error querying Gemini: {e}")
        return None

# --- Agent 1: Fact Checker ---
fact_system = """You are FactCheckerAgent. 
Classify the review as: 'real', 'fake(misinformation)', or 'unrelated'.
Output ONLY JSON: {"label": "..."}"""

# --- Agent 2: Sentiment Analyzer ---
sentiment_system = """You are SentimentAgent. 
Analyze sentiment as: 'positive', 'negative', or 'neutral'.
Output ONLY JSON: {"sentiment": "..."}"""

# --- Agent 3: Combiner ---
combiner_system = """You are CombinerAgent. 
Review the provided labels and text. Correct any obvious logical errors.
Output ONLY JSON: {"label": "...", "sentiment": "..."}"""

In [ ]:
data_path = "/content/drive/MyDrive/Czech/M12_final_v1_to_dataset.jsonl"

reviews = []
with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.strip():
            try:
                reviews.append(json.loads(line))
            except:
                pass
        if len(reviews) >= 100: # Testing cap
            break

results = []

for row in tqdm(reviews, desc="3-Agent Gemini Processing"):
    text = row.get("text", "")
    if not text.strip():
        label, sentiment = "unrelated", "neutral"
    else:
        # Agent 1: Fact Check
        fact_res = query_gemini_agent(fact_system, f"Text: {text}")
        label = fact_res.get("label", "unrelated") if fact_res else "unrelated"

        # Agent 2: Sentiment
        sent_res = query_gemini_agent(sentiment_system, f"Text: {text}")
        sentiment = sent_res.get("sentiment", "neutral") if sent_res else "neutral"

        # Agent 3: Final Combiner
        combiner_input = f"Text: {text}\nLabel: {label}\nSentiment: {sentiment}"
        final_res = query_gemini_agent(combiner_system, combiner_input)
        
        if final_res:
            label = final_res.get("label", label)
            sentiment = final_res.get("sentiment", sentiment)

    results.append({
        **row, # Keep original data
        "label": label,
        "sentiment": sentiment
    })

In [ ]:
df = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/Czech/gemini-analyzed_reviews.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Done! Saved to {output_path}")
files.download(output_path)